# N088 · 强连通分量与缩点

**目标：** 把有向循环压缩为DAG并追踪安全状态。

**先修：** N080, N082。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** Kosaraju；Tarjan lowlink；栈内标记；缩点；终止状态。

## 从问题到算法

强连通分量是相互可达的最大顶点集合。Kosaraju通过完成时间与反图分离分量；Tarjan把尚未归属的点放在活动栈中，用low记录能回到的活动祖先。注意：访问过但已经离开活动栈的点，不可用于降低low。两种实现都用显式DFS帧。

## 最小实现

**本章接口：** `kosaraju_scc(adj)`、`tarjan_scc(adj)`、`condensation_graph(adj, component)`

In [1]:
def kosaraju_scc(adj):
    n=len(adj); seen=set(); order=[]
    for s in range(n):
        if s in seen: continue
        seen.add(s); stack=[(s,iter(adj[s]))]
        while stack:
            u,it=stack[-1]
            try: v=next(it)
            except StopIteration: order.append(u); stack.pop(); continue
            if v not in seen: seen.add(v); stack.append((v,iter(adj[v])))
    rev=[[] for _ in range(n)]
    for u,a in enumerate(adj):
        for v in a: rev[v].append(u)
    component=[-1]*n; label=0
    for s in reversed(order):
        if component[s]!=-1: continue
        component[s]=label; stack=[s]
        while stack:
            for v in rev[stack.pop()]:
                if component[v]==-1: component[v]=label; stack.append(v)
        label+=1
    return component

def tarjan_scc(adj):
    n=len(adj); disc=[-1]*n; low=[0]*n; active=[]; on=[False]*n; comp=[-1]*n; clock=label=0
    for s in range(n):
        if disc[s]!=-1: continue
        disc[s]=low[s]=clock; clock+=1; active.append(s); on[s]=True
        frames=[(s,iter(adj[s]),-1)]
        while frames:
            u,it,parent=frames[-1]
            try: v=next(it)
            except StopIteration:
                frames.pop()
                if parent!=-1: low[parent]=min(low[parent],low[u])
                if low[u]==disc[u]:
                    while True:
                        v=active.pop(); on[v]=False; comp[v]=label
                        if v==u: break
                    label+=1
                continue
            if disc[v]==-1:
                disc[v]=low[v]=clock; clock+=1; active.append(v); on[v]=True
                frames.append((v,iter(adj[v]),u))
            elif on[v]: low[u]=min(low[u],disc[v])
    return comp

def condensation_graph(adj,component):
    result=[set() for _ in range(max(component,default=-1)+1)]
    for u,a in enumerate(adj):
        for v in a:
            if component[u]!=component[v]: result[component[u]].add(component[v])
    return [sorted(a) for a in result]

## 正确性、前提与复杂度

Tarjan在low[u]=disc[u]时，从活动栈顶到u恰构成一个SCC：它们可沿DFS树到达，且未能到达更早的活动祖先。缩点图若有环，则环上分量相互可达，本应合并，故缩点图必为DAG。分量编号只是标签，不比较具体编号。

**代价：** 两算法O(V+E)时间、O(V+E)辅助空间（Kosaraju保存反图）；缩点去重平均O(V+E)，排序输出另加各分量出边排序代价。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

adj=[[1],[2,3],[0],[4],[3],[]]; comp=tarjan_scc(adj)
show_table(['顶点','邻居','分量'],[(i,a,comp[i]) for i,a in enumerate(adj)])
print('缩点图：',condensation_graph(adj,comp))

顶点,邻居,分量
0,[1],1
1,"[2, 3]",1
2,[0],1
3,[4],0
4,[3],0
5,[],2


缩点图： [[], [0], []]


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from itertools import product
for bits in product(range(2),repeat=9):
    adj=[[v for v in range(3) if bits[u*3+v]] for u in range(3)]
    reach=[[i==j or j in adj[i] for j in range(3)] for i in range(3)]
    for k in range(3):
        for i in range(3):
            for j in range(3): reach[i][j]|=reach[i][k] and reach[k][j]
    a=kosaraju_scc(adj); b=tarjan_scc(adj)
    for i in range(3):
        for j in range(3): assert (a[i]==a[j])==(b[i]==b[j])==(reach[i][j] and reach[j][i])
    dag=condensation_graph(adj,b)
    def walk(u,path):
        assert u not in path
        for v in dag[u]: walk(v,path|{u})
    for u in range(len(dag)): walk(u,set())
chain=[[i+1] for i in range(1499)]+[[]]
assert len(set(tarjan_scc(chain)))==1500
assert tarjan_scc([])==kosaraju_scc([])==[]
print('N088: 所有本章断言通过')

N088: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 以反向拓扑法作802对照。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 解释不在栈内节点不能直接更新Tarjan回边值。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 802. Find Eventual Safe States（extension）：[官方入口](https://leetcode.com/problems/find-eventual-safe-states/)

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。